# 09 — Evaluación final

Este notebook evalúa en el conjunto **test** el VLA seleccionado y el baseline MLP. La evaluación es offline: compara las acciones predichas con las registradas en las demostraciones, sin ejecutar acciones en un robot o simulador.

El modelo y el umbral de `terminate` se han seleccionado previamente con validation. Por ello, este notebook no ajusta ningún parámetro usando test.

## 1. Configuración e imports

Se fijan las rutas del proyecto y se importan las funciones compartidas. La ejecución funciona desde la carpeta `notebooks` o desde la raíz del proyecto.

In [1]:
from pathlib import Path
import json
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT_DIR = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'src').exists() else NOTEBOOK_DIR
if not (ROOT_DIR / 'src').exists():
    raise FileNotFoundError('No se ha encontrado la carpeta src del proyecto.')
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.evaluation import benchmark_inference, evaluate_actions
from src.project_config import CACHE_DIR, PROJECT_DIR, experiment_dirs
from src.vla_model import VLA

SEED = 42
BATCH_SIZE = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path(PROJECT_DIR)
OUTPUT_DIRS = experiment_dirs('evaluacion_final')
RESULTS_OUTPUT_DIR = OUTPUT_DIRS['results']
VLA_RESULT_PATH = PROJECT_DIR / 'results' / 'experimentos_vla' / 'vla_referencia' / 'resultado.json'
BASELINE_RESULT_PATH = PROJECT_DIR / 'results' / 'baseline_mlp_binary_phase1' / 'baseline_mlp_binary_validation.json'
NORMALIZATION_PATH = PROJECT_DIR / 'data' / 'parametros_normalizacion.json'

print(f'Proyecto: {PROJECT_DIR}')
print(f'Dispositivo: {DEVICE}')

Proyecto: C:\TFM_Codigo\TFM-VLA
Dispositivo: cpu


## 2. Carga del conjunto de test

Se cargan los embeddings de CLIP y las acciones normalizadas. Test se usa únicamente para esta evaluación final.

In [2]:
def cargar_particion_test():
    ruta = CACHE_DIR / 'test.npz'
    if not ruta.exists():
        raise FileNotFoundError(f'No existe {ruta}. Ejecuta antes 04_clip_embeddings.ipynb.')
    with np.load(ruta) as datos:
        claves = ('imagenes_static', 'imagenes_gripper', 'textos', 'acciones')
        if not set(claves).issubset(datos.files):
            raise KeyError(f'El archivo test debe contener: {claves}')
        arrays = tuple(np.asarray(datos[clave], dtype=np.float32) for clave in claves)
    static, gripper, texto, acciones = arrays
    if not (len(static) == len(gripper) == len(texto) == len(acciones)):
        raise ValueError('Las entradas y acciones de test tienen longitudes distintas.')
    if static.shape[1] != 512 or gripper.shape[1] != 512 or texto.shape[1] != 512 or acciones.shape[1] != 8:
        raise ValueError('Las dimensiones de test no coinciden con la configuración esperada.')
    return arrays

test_arrays = cargar_particion_test()
test_dataset = TensorDataset(*(torch.from_numpy(array) for array in test_arrays))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

with NORMALIZATION_PATH.open(encoding='utf-8') as archivo:
    normalizacion = json.load(archivo)
minimum = np.asarray(normalizacion['minimo'], dtype=np.float32)
scale = np.asarray(normalizacion['escala'], dtype=np.float32)

print(f'Muestras de test: {len(test_dataset):,}')
print(f'Embeddings static: {test_arrays[0].shape} | texto: {test_arrays[2].shape}')

Muestras de test: 23,826
Embeddings static: (23826, 512) | texto: (23826, 512)


## 3. Carga de los modelos finales

El VLA de referencia se recupera con su configuración del notebook 08. El baseline se recupera desde el resultado del notebook 05. Ambos umbrales de `terminate` proceden de validation.

In [3]:
def cargar_resultado(ruta, notebook_origen):
    if not ruta.exists():
        raise FileNotFoundError(f'No existe {ruta}. Ejecuta antes {notebook_origen}.')
    with ruta.open(encoding='utf-8') as archivo:
        return json.load(archivo)

vla_info = cargar_resultado(VLA_RESULT_PATH, '08_experimentos.ipynb')
baseline_info = cargar_resultado(BASELINE_RESULT_PATH, '05_baseline_mlp.ipynb')

class BaselineMLP(nn.Module):
    # Misma arquitectura sencilla utilizada en el notebook 05.
    def __init__(self, input_dim=1536, hidden_dim_1=512, hidden_dim_2=256, output_dim=8, dropout=0.10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim_1, hidden_dim_2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, output_dim), nn.Sigmoid(),
        )

    def forward(self, features):
        return self.network(features)

def cargar_checkpoint(modelo, ruta):
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f'No existe el checkpoint {ruta}.')
    checkpoint = torch.load(ruta, map_location=DEVICE, weights_only=False)
    modelo.load_state_dict(checkpoint['model_state_dict'])
    return modelo.to(DEVICE).eval()

vla_model = VLA(
    clip_encoder=None, embedding_dim=512, fusion_dim=vla_info['fusion_dim'],
    decoder_hidden_dim=128, num_layers=vla_info['num_layers'],
).to(DEVICE)
vla_model = cargar_checkpoint(vla_model, vla_info['checkpoint'])
baseline_model = cargar_checkpoint(BaselineMLP(), baseline_info['artifacts']['checkpoint'])

vla_threshold = float(vla_info['terminate_threshold'])
baseline_threshold = float(baseline_info['terminate_threshold'])
print(f"VLA: {vla_info['name']} | épocas ejecutadas: {vla_info['epochs_executed']} | umbral: {vla_threshold:.2f}")
print(f"Baseline: épocas ejecutadas: {baseline_info['epochs_executed']} | umbral: {baseline_threshold:.2f}")

VLA: vla_referencia | épocas ejecutadas: 3 | umbral: 0.66
Baseline: épocas ejecutadas: 20 | umbral: 0.91


## 4. Predicciones sobre test

Las predicciones se calculan por lotes para no cargar todas las operaciones en memoria a la vez. El baseline concatena las tres representaciones de CLIP; el VLA las procesa como modalidades separadas.

In [4]:
@torch.no_grad()
def predecir_test(modelo, tipo_modelo):
    predicciones, objetivos = [], []
    for static, gripper, texto, acciones in test_loader:
        static, gripper, texto = static.to(DEVICE), gripper.to(DEVICE), texto.to(DEVICE)
        if tipo_modelo == 'vla':
            salida = modelo(static_embeddings=static, gripper_embeddings=gripper, text_embeddings=texto)
        else:
            # El MLP recibe un único vector con los tres embeddings concatenados.
            salida = modelo(torch.cat([static, gripper, texto], dim=1))
        predicciones.append(salida.cpu())
        objetivos.append(acciones)
    return torch.cat(predicciones).numpy(), torch.cat(objetivos).numpy()

vla_predictions, targets = predecir_test(vla_model, 'vla')
baseline_predictions, baseline_targets = predecir_test(baseline_model, 'baseline')
if not np.array_equal(targets, baseline_targets):
    raise RuntimeError('Los objetivos de test no coinciden entre ambos modelos.')
print(f'Predicciones generadas: {vla_predictions.shape}')

Predicciones generadas: (23826, 8)


## 5. Métricas de precisión

Se calculan MAE y RMSE para las seis componentes continuas, similitud del coseno para el desplazamiento `xyz` en escala original y métricas de clasificación para `terminate` y la pinza.

In [5]:
vla_metrics = evaluate_actions(vla_predictions, targets, minimum, scale, terminate_threshold=vla_threshold)
baseline_metrics = evaluate_actions(baseline_predictions, targets, minimum, scale, terminate_threshold=baseline_threshold)

def fila_metricas(nombre, metricas):
    return {
        'modelo': nombre,
        'MAE': metricas['continuous_normalized']['mae'],
        'RMSE': metricas['continuous_normalized']['rmse'],
        'coseno_xyz': metricas['xyz_denormalized_cosine_similarity'],
        'terminate_accuracy': metricas['terminate']['accuracy'],
        'terminate_F1': metricas['terminate']['f1'],
        'gripper_accuracy': metricas['gripper']['accuracy'],
        'gripper_F1': metricas['gripper']['f1'],
    }

precision_table = pd.DataFrame([
    fila_metricas('VLA referencia', vla_metrics),
    fila_metricas('Baseline MLP', baseline_metrics),
]).set_index('modelo')
display(precision_table.round(4))

component_table = pd.DataFrame({
    'MAE VLA': vla_metrics['continuous_normalized']['mae_by_component'],
    'MAE baseline': baseline_metrics['continuous_normalized']['mae_by_component'],
    'RMSE VLA': vla_metrics['continuous_normalized']['rmse_by_component'],
    'RMSE baseline': baseline_metrics['continuous_normalized']['rmse_by_component'],
})
display(component_table.round(4))

,MAE,RMSE,coseno_xyz,terminate_accuracy,terminate_F1,gripper_accuracy,gripper_F1
modelo,,,,,,,
VLA referencia,0.0346,0.0516,0.0572,0.9600,0.0498,0.8312,0.7884
Baseline MLP,0.0333,0.0487,0.3186,0.9699,0.2077,0.8899,0.8681


,MAE VLA,MAE baseline,RMSE VLA,RMSE baseline
x,0.0275,0.0265,0.0410,0.0381
y,0.0453,0.0413,0.0649,0.0584
z,0.0468,0.0450,0.0682,0.0636
rx,0.0180,0.0182,0.0273,0.0274
ry,0.0301,0.0298,0.0442,0.0435
rz,0.0400,0.0389,0.0639,0.0614


## 6. Eficiencia computacional

La inferencia se mide usando los embeddings ya cacheados, que es la misma condición para ambos modelos. Los tiempos de entrenamiento se recuperan de los resultados de los notebooks anteriores.

In [6]:
N_SAMPLES_TIME = min(32, len(test_dataset))
static_time = torch.from_numpy(test_arrays[0][:N_SAMPLES_TIME]).to(DEVICE)
gripper_time = torch.from_numpy(test_arrays[1][:N_SAMPLES_TIME]).to(DEVICE)
text_time = torch.from_numpy(test_arrays[2][:N_SAMPLES_TIME]).to(DEVICE)

vla_inference = benchmark_inference(
    lambda: vla_model(static_embeddings=static_time, gripper_embeddings=gripper_time, text_embeddings=text_time),
    N_SAMPLES_TIME,
)
baseline_inference = benchmark_inference(
    lambda: baseline_model(torch.cat([static_time, gripper_time, text_time], dim=1)),
    N_SAMPLES_TIME,
)

def parametros(modelo):
    return sum(p.numel() for p in modelo.parameters()), sum(p.numel() for p in modelo.parameters() if p.requires_grad)

vla_total, vla_trainable = parametros(vla_model)
baseline_total, baseline_trainable = parametros(baseline_model)
efficiency_table = pd.DataFrame([
    {'modelo': 'VLA referencia', 'parámetros_totales': vla_total, 'parámetros_entrenables': vla_trainable,
     'ms_por_muestra': vla_inference['mean_ms_per_sample'], 'desviación_ms': vla_inference['std_ms_per_sample'],
     'tiempo_entrenamiento_s': vla_info['total_training_time_seconds'],
     'tiempo_medio_época_s': vla_info['total_training_time_seconds'] / vla_info['epochs_executed']},
    {'modelo': 'Baseline MLP', 'parámetros_totales': baseline_total, 'parámetros_entrenables': baseline_trainable,
     'ms_por_muestra': baseline_inference['mean_ms_per_sample'], 'desviación_ms': baseline_inference['std_ms_per_sample'],
     'tiempo_entrenamiento_s': baseline_info['total_training_time_seconds'],
     'tiempo_medio_época_s': baseline_info['total_training_time_seconds'] / baseline_info['epochs_executed']},
]).set_index('modelo')
display(efficiency_table.round(3))

,parámetros_totales,parámetros_entrenables,ms_por_muestra,desviación_ms,tiempo_entrenamiento_s,tiempo_medio_época_s
modelo,,,,,,
VLA referencia,1352072,1352072,0.095,0.015,1252.632,417.544
Baseline MLP,920328,920328,0.023,0.003,642.272,32.114


## 7. Guardado de resultados

Se guarda un resumen en CSV y un resultado detallado en JSON. El notebook 10 utilizará estos archivos para generar las tablas y gráficas de la memoria.

In [7]:
comparison_table = precision_table.join(efficiency_table)
csv_path = RESULTS_OUTPUT_DIR / 'comparacion_test.csv'
json_path = RESULTS_OUTPUT_DIR / 'evaluacion_test.json'
comparison_table.to_csv(csv_path)

resultados = {
    'scope': 'evaluación offline sobre test',
    'seed': SEED,
    'device': str(DEVICE),
    'test_samples': len(test_dataset),
    'provisional': bool(vla_info['epochs_executed'] < 50),
    'vla': {'configuration': vla_info, 'test_metrics': vla_metrics, 'inference_cached': vla_inference},
    'baseline': {'configuration': baseline_info, 'test_metrics': baseline_metrics, 'inference_cached': baseline_inference},
    'comparison': comparison_table.reset_index().to_dict(orient='records'),
}
with json_path.open('w', encoding='utf-8') as archivo:
    json.dump(resultados, archivo, indent=2, ensure_ascii=False)

estado = 'provisionales (entrenamiento corto)' if resultados['provisional'] else 'finales'
print(f'Resultados {estado}.')
print(f'CSV: {csv_path}')
print(f'JSON: {json_path}')

Resultados provisionales (entrenamiento corto).
CSV: C:\TFM_Codigo\TFM-VLA\results\evaluacion_final\comparacion_test.csv
JSON: C:\TFM_Codigo\TFM-VLA\results\evaluacion_final\evaluacion_test.json


## 8. Cierre

La evaluación compara precisión y coste computacional sin combinarlos en una única puntuación. Tras completar el entrenamiento definitivo, basta con volver a ejecutar este notebook para actualizar los resultados de test y continuar con el notebook 10.